# Qwen QLoRA for Mathematical Reasoning

This pipeline compares an unchanged Qwen2.5-1.5B-Instruct baseline with answer-only and worked-solution adapters. All three conditions use the same prompt, greedy decoding, and final-line numerical scoring.


In [ ]:
# Set cache locations before importing model and dataset libraries.
from pathlib import Path
import os
import importlib.metadata as metadata

WORK_DIR = os.environ.get("MATH_WORK_DIR", str(Path.cwd()))
CACHE_ROOT = Path(WORK_DIR) / "cache"
for key, folder in {
    "HF_HOME": "huggingface",
    "HF_HUB_CACHE": "huggingface/hub",
    "HF_DATASETS_CACHE": "huggingface/datasets",
    "PIP_CACHE_DIR": "pip",
    "TMPDIR": "tmp",
    "JUPYTER_RUNTIME_DIR": "jupyter/runtime",
}.items():
    path = CACHE_ROOT / folder
    path.mkdir(parents=True, exist_ok=True)
    os.environ[key] = str(path)

packages = ("torch", "transformers", "datasets", "bitsandbytes", "accelerate",
            "ipykernel", "scikit-learn", "pandas")
PACKAGE_VERSIONS = {name: metadata.version(name) for name in packages}
print("Python packages:", PACKAGE_VERSIONS)
print("Work directory:", WORK_DIR)


In [ ]:
"""
Import Related Modules
"""

import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
import os
from datasets import load_dataset
import json
from torch.utils.data import Dataset, DataLoader
import re
from tqdm import tqdm
import re
from decimal import Decimal, InvalidOperation
from fractions import Fraction
import torch.nn as nn
from transformers import BitsAndBytesConfig
import time
import gc
import random


In [ ]:
import hashlib

CKPT_DIR = os.path.join(WORK_DIR, "ckpt")
LOG_DIR = os.path.join(WORK_DIR, "logs")
SUBMIT_DIR = os.path.join(WORK_DIR, "submission")
ANALYSIS_DIR = os.path.join(WORK_DIR, "analysis")
DATA_DIR = os.path.join(WORK_DIR, "data")
for folder in (CKPT_DIR, LOG_DIR, SUBMIT_DIR, ANALYSIS_DIR, DATA_DIR):
    os.makedirs(folder, exist_ok=True)

def atomic_json(path, data):
    temporary = str(path) + ".tmp"
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(data, handle, indent=2, ensure_ascii=False, allow_nan=False)
    os.replace(temporary, path)

def digest(data):
    return hashlib.sha256(json.dumps(data, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def file_sha256(path):
    value = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            value.update(block)
    return value.hexdigest()

SEED = 42
NUM_EPOCHS, MAX_SEQ_LENGTH, LEARNING_RATE = 2, 512, 1e-5
LORA_RANK, LORA_ALPHA, LORA_DROPOUT = 8, 16, 0.1
LORA_TARGET_MODULES = None
TRAIN_BATCH_SIZE, EVAL_BATCH_SIZE, MAX_NEW_TOKENS = 2, 8, 256

def seed_all(seed=SEED):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all()
assert torch.cuda.is_available(), "This notebook requires a CUDA GPU."
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.benchmark = False
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
provenance_path = os.path.join(LOG_DIR, "provenance.json")
notebook_path = Path(WORK_DIR) / "math_reasoning.ipynb"
with open(notebook_path, encoding="utf-8") as handle:
    notebook_sources = ["".join(c["source"]) if isinstance(c["source"], list) else c["source"]
                       for c in json.load(handle)["cells"] if c["cell_type"] == "code"]
CODE_HASH = digest(notebook_sources)
if os.path.exists(provenance_path):
    with open(provenance_path, encoding="utf-8") as handle:
        provenance = json.load(handle)
    assert provenance["code_hash"] == CODE_HASH, "Code changed: use a separate work directory."
    assert provenance["packages"] == PACKAGE_VERSIONS, "Package versions changed: use a separate run."
else:
    provenance = {
        "model": MODEL_ID,
        "model_revision": '989aa7980e4cf806f80c7fef2b1adb7bc71aa306',
        "gsm8k_revision": '740312add88f781978c0658806c59bc2815b9866',
        "svamp_revision": '5e0bf1e5e7c0e9c4bc39180d224f41f3f801b7ef',
        "code_hash": CODE_HASH,
        "packages": PACKAGE_VERSIONS,
        "gpu": torch.cuda.get_device_name(0),
        "torch_cuda": torch.version.cuda,
    }
    atomic_json(provenance_path, provenance)
MODEL_REVISION = provenance["model_revision"]
print(provenance)


## Dataset snapshots

GSM8K and SVAMP are loaded at fixed revisions and exported locally for the run.


In [ ]:
gsm8k = load_dataset("openai/gsm8k", "main", revision=provenance["gsm8k_revision"])
svamp = load_dataset("ChilleD/SVAMP", revision=provenance["svamp_revision"])
for name, dataset in (("gsm8k", gsm8k), ("svamp", svamp)):
    folder = os.path.join(DATA_DIR, name)
    os.makedirs(folder, exist_ok=True)
    for split in ("train", "test"):
        dataset[split].to_json(os.path.join(folder, f"{split}.jsonl"))
print("Dataset snapshots saved:", DATA_DIR)


In [ ]:
def read_lines(file_name):
  """
  Helper functions, returns a list of dicts
  """
  res = []
  with open(file_name, 'r') as f:
    for line in f:
      res.append(json.loads(line))
  return res


## Fixed training and validation split

Seed 42 creates 6,725 GSM8K training examples and 748 validation examples. GSM8K test and all 1,000 SVAMP examples serve as final evaluation sets.


In [ ]:
gsm8k_train = read_lines(os.path.join(DATA_DIR, 'gsm8k/train.jsonl'))
gsm8k_test = read_lines(os.path.join(DATA_DIR, 'gsm8k/test.jsonl'))
svamp_train = read_lines(os.path.join(DATA_DIR, 'svamp/train.jsonl'))
svamp_test = read_lines(os.path.join(DATA_DIR, 'svamp/test.jsonl'))
from sklearn.model_selection import train_test_split
SEED = 42
val_ratio = 0.1
torch.manual_seed(SEED)
random.seed(SEED)
official_train = gsm8k_train
train_indices, val_indices = train_test_split(
    list(range(len(official_train))), test_size=val_ratio, random_state=SEED
)
gsm8k_train = [official_train[i] for i in train_indices]
gsm8k_val = [official_train[i] for i in val_indices]
assert set(train_indices).isdisjoint(val_indices)
svamp_data = svamp_train + svamp_test
split_manifest = {
    "train_indices": train_indices,
    "val_indices": val_indices,
    "dataset_revisions": {k: provenance[k] for k in ("gsm8k_revision", "svamp_revision")},
    "official_train_sha256": file_sha256(os.path.join(DATA_DIR, "gsm8k/train.jsonl")),
    "seed": SEED,
    "val_ratio": val_ratio,
    "gsm8k_train_size": len(gsm8k_train),
    "gsm8k_val_size": len(gsm8k_val),
    "gsm8k_test_size": len(gsm8k_test),
    "svamp_total_size": len(svamp_data),
    "gsm8k_source": "openai/gsm8k (main)",
    "svamp_source": "ChilleD/SVAMP",
    "split_method": "sklearn.model_selection.train_test_split",
}

split_manifest_path = os.path.join(LOG_DIR, "split_manifest.json")
with open(split_manifest_path, "w", encoding="utf-8") as f:
    json.dump(split_manifest, f, indent=2, ensure_ascii=False)
print(f"saved: {split_manifest_path}")
print(f"gsm8k_train: {len(gsm8k_train)}")
print(f"gsm8k_val:   {len(gsm8k_val)}")
print(f"gsm8k_test:  {len(gsm8k_test)}")
assert len(svamp_data) == 1000, "SVAMP must contain all 1,000 examples."
SPLIT_HASH = digest(split_manifest)


## Chat formatting and supervision

The two adapters differ in their supervised targets: a final answer line versus a worked solution followed by that line. Source tokens are excluded from the training loss.


In [ ]:
INSTRUCTION = '''Please solve the following problem. End your response with a final line in exactly this format:
#### <answer>
where <answer> is a single number (it may be negative, contain commas, be a decimal, or a simple fraction like 3/4). Do not write anything after this line.\n\n'''


class GSM8K_Dataset(Dataset):
  """
  Wraps the GSM8K dataset into (source, target) training/evaluation
  examples, formatted with the Qwen chat template.

  Depending on `mode`, the supervised target differs:
    - 'sft_answer':    target is only the final "#### <answer>" line
    - 'sft_reasoning' or 'default': target is the full worked solution ending in "#### <answer>"
  """
  def __init__(self, data, tokenizer=None, mode='default', max_len=512):
    """
    Args:
        data: a list of dicts, each with 'question' and 'answer' fields
              (raw GSM8K format).
        tokenizer: the tokenizer used to build the chat-template prompt
                   and to tokenize source/target for training.
        mode: 'sft_answer', 'sft_reasoning', or 'default'. Controls which
              target format is constructed for each example.
    """
    self.mode = mode
    self.tokenizer = tokenizer
    self.data = self.prepare_data(data)
    self.max_len = max_len

  def prepare_data(self, data):
    """
    Build (source, target) pairs for every example in `data`.

    `source` is the user prompt wrapped with the chat template, ending
    right before the assistant's turn (add_generation_prompt=True), so
    it's ready to either be fed to generate() or concatenated with a
    target for training.

    `target` is the supervised text the assistant should produce,
    format depending on self.mode.
    """
    processed_data = []
    for idx, dat in enumerate(data):
      raw_question = dat['question']
      raw_response = dat['answer']
      # if supervise on reasoning steps
      question = INSTRUCTION + raw_question
      source = self.tokenizer.apply_chat_template(
          [{"role": "user", "content": question}],
          tokenize=False,
          add_generation_prompt=True
      )
      if self.mode == 'sft_answer':
        processed_data.append(
            {
                "example_id": f"gsm8k-{idx:05d}",
                "source": source,
                "target": self.get_final_answer(raw_response)
            }
        )
      elif self.mode == 'sft_reasoning' or self.mode == 'default':
        processed_data.append(
            {
                "example_id": f"gsm8k-{idx:05d}",
                "source": source,
                "target": self.get_reasoning(raw_response)
            }
        )
      else:
        raise NotImplementedError
    return processed_data

  def get_final_answer(self, raw_answer):
    """
    Extract just the final numeric answer from the raw GSM8K 'answer'
    field and re-wrap it in the required "#### <answer>" format.

    Used for the answer-only (SFT-Answer) supervision target.
    """
    # GSM8K answers end with a line such as `#### 42`. Keep only that
    # final-answer line for answer-only supervision.
    final_answer = raw_answer.rsplit("####", 1)[-1].strip()
    return f"#### {final_answer}"
  def get_reasoning(self, raw_answer):
    """
    Extract the reasoning steps from the raw GSM8K 'answer' field.

    You're welcome to apply additional cleaning procedure.
    """
    # The raw GSM8K answer already contains the worked solution and
    # its final `#### <answer>` line.
    return raw_answer.strip()

  def __len__(self):
    return len(self.data)

  def __getitem__(self, index):
    """
    Tokenize one (source, target) pair and build the loss mask.

    Returns a dict with input_ids/attention_mask/labels ready to be
    batched by collate_fn. `labels` marks the source (prompt) portion
    with -100 so the loss is only computed on the target (assistant)
    portion — this is what makes it "supervised fine-tuning on the
    target" rather than training the model to also predict the prompt.
    """
    data = self.data[index]
    source = data['source']
    target = data['target']
    example_id = data['example_id']
    full_text = source + target + self.tokenizer.eos_token
    full_ids = self.tokenizer(full_text, add_special_tokens=False)["input_ids"]
    source_ids = self.tokenizer(source, add_special_tokens=False)["input_ids"]

    if len(full_ids) > self.max_len:
        dropped = len(full_ids) - self.max_len
        full_ids = full_ids[dropped:]
        supervised_start = max(0, len(source_ids) - dropped)
    else:
        supervised_start = len(source_ids)

    labels = [-100] * supervised_start + full_ids[supervised_start:]

    return {
        "source": source,
        "example_id": example_id,
        "target": target,
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    }


def collate_fn(batch):
  """
  Collate a list of __getitem__ outputs into a single padded batch.

  Uses left-padding. `input_ids` is padded with pad_token_id and
  `attention_mask` with 0, but `labels` must be padded with -100
  (NOT any real token id) so that padding positions are also excluded
  from the loss — for the same reason the source/prompt portion was
  masked with -100 in __getitem__.
  """
  source = [item['source'] for item in batch]
  target = [item['target'] for item in batch]
  input_ids = [item['input_ids'] for item in batch]
  attention_mask = [item['attention_mask'] for item in batch]
  labels = [item['labels'] for item in batch]
  example_ids = [item['example_id'] for item in batch]
  max_len = max(len(ids) for ids in input_ids)
  input_ids = [[tokenizer.pad_token_id] * (max_len - len(ids)) + ids for ids in input_ids]
  attention_mask = [[0] * (max_len - len(mask)) + mask for mask in attention_mask]
  labels = [[-100] * (max_len - len(values)) + values for values in labels]

  return {
      "example_id": example_ids,
      "source": source,
      "target": target,
      "input_ids": torch.tensor(input_ids, dtype=torch.long),
      "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
      "labels": torch.tensor(labels, dtype=torch.long),
  }


class SVAMP_Dataset(Dataset):
  def __init__(self, data, tokenizer=None):
    self.tokenizer = tokenizer
    self.data = self.prepare_data(data)
    self.max_len = 1024

  def prepare_data(self, data):
    processed_data = []
    for dat in data:
      body = str(dat['Body']).strip()
      question_text = str(dat['Question']).strip()
      raw_answer = dat['Answer']
      example_id = dat['ID']

      question = INSTRUCTION + f"{body} {question_text}".strip()
      source = self.tokenizer.apply_chat_template(
          [{"role": "user", "content": question}],
          tokenize=False,
          add_generation_prompt=True
      )
      processed_data.append(
          {
              "ID": example_id,
              "source": source,
              "target": self.get_final_answer(raw_answer)
          }
      )
    return processed_data

  def get_final_answer(self, raw_answer):
    return f"#### {raw_answer}"

  def __len__(self):
    return len(self.data)

  def __getitem__(self, index):
    # returns a <source, target> pair
    data = self.data[index]
    source = data['source']
    target = data['target']
    full_text = source + target + self.tokenizer.eos_token
    full_ids = self.tokenizer(full_text, add_special_tokens=False)["input_ids"]
    source_ids = self.tokenizer(source, add_special_tokens=False)["input_ids"]

    assert full_ids[:len(source_ids)] == source_ids

    if len(full_ids) > self.max_len:
        dropped = len(full_ids) - self.max_len
        full_ids = full_ids[dropped:]
        supervised_start = max(0, len(source_ids) - dropped)
    else:
        supervised_start = len(source_ids)

    labels = [-100] * supervised_start + full_ids[supervised_start:]



    return {
        "source": source,
        "target": target,
        "example_id": data['ID'],
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    }


## Quantized model

The frozen base uses NF4, double quantization, and FP16 computation.


In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
seed_all()
lm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, attn_implementation="sdpa",
    device_map={"": 0}, quantization_config=quant_config, torch_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
print("Model:", MODEL_ID, MODEL_REVISION)
print("GPU:", torch.cuda.get_device_name(0), "| dtype:", lm.dtype)


In [ ]:
# =========================================================
# Prepare all dataset
# =========================================================

gsm8k_train_dataset_answer_only = GSM8K_Dataset(gsm8k_train, tokenizer, mode='sft_answer')
gsm8k_val_dataset_answer_only = GSM8K_Dataset(gsm8k_val, tokenizer, mode='sft_answer')

gsm8k_train_dataset_reasoning = GSM8K_Dataset(gsm8k_train, tokenizer, mode='sft_reasoning')
gsm8k_val_dataset_reasoning = GSM8K_Dataset(gsm8k_val, tokenizer, mode='sft_reasoning')

gsm8k_test_dataset_default = GSM8K_Dataset(gsm8k_test, tokenizer, mode='default')

svamp_dataset = SVAMP_Dataset(svamp_data, tokenizer)


In [ ]:
# Dataset format checks: these do not train or generate anything.
# They verify the contracts used by the later training/evaluation cells.
_sample = gsm8k_train_dataset_answer_only[0]
assert {"source", "target", "input_ids", "attention_mask", "labels"}.issubset(_sample)
assert len(_sample["input_ids"]) == len(_sample["attention_mask"]) == len(_sample["labels"])
assert _sample["target"].lstrip().startswith("####")
_target_ids = tokenizer(_sample["target"] + tokenizer.eos_token, add_special_tokens=False)["input_ids"]
assert _sample["labels"][-len(_target_ids):] == _target_ids
assert -100 in _sample["labels"]

_batch = collate_fn([gsm8k_train_dataset_answer_only[0], gsm8k_train_dataset_answer_only[1]])
assert _batch["input_ids"].ndim == 2
assert _batch["input_ids"].shape == _batch["attention_mask"].shape == _batch["labels"].shape
assert _batch["input_ids"].shape[0] == 2
assert torch.all((_batch["attention_mask"] == 0) | (_batch["attention_mask"] == 1))
assert torch.all(_batch["labels"][_batch["attention_mask"] == 0] == -100)
print("Dataset format checks passed.")


## Numerical answer extraction

A valid output ends with exactly one `#### <number>` line. Signs, commas, decimals, and fractions are normalized to exact rational values. Extraction failures count as incorrect predictions.


In [ ]:
NUMBER_PATTERN = r"[+-]?(?:(?:\d[\d,]*)(?:\.\d*)?|\.\d+)(?:\s*/\s*[+-]?(?:(?:\d[\d,]*)(?:\.\d*)?|\.\d+))?"
FINAL_ANSWER_RE = re.compile(rf"^\s*####\s*({NUMBER_PATTERN})\s*$")
MARKER_RE = re.compile(r"^\s*####")

def normalize_number(text):
    """
    Convert a string-form number into an exact Fraction, so it can be
    compared reliably against the gold answer. Returns None on failure.

    Must handle, at minimum:
      - optional leading sign (+/-)
      - thousands separators (commas), e.g. "1,200"
      - decimals, e.g. "3.14", ".5"
      - simple fractions, e.g. "3/4", with each side possibly signed/decimal
      - division by zero, or any unparseable input -> return None
        (never raise an exception out of this function)
    """
    if text is None:
        return None
    candidate = str(text).strip()
    if not re.fullmatch(NUMBER_PATTERN, candidate):
        return None

    try:
        if "/" in candidate:
            numerator_text, denominator_text = re.split(r"\s*/\s*", candidate, maxsplit=1)
            numerator = Fraction(Decimal(numerator_text.replace(",", "")))
            denominator = Fraction(Decimal(denominator_text.replace(",", "")))
            return numerator / denominator if denominator else None
        return Fraction(Decimal(candidate.replace(",", "")))
    except (InvalidOperation, ValueError, ZeroDivisionError):
        return None


def extract_final_answer(generation):
    """
    Extract the answer from the '#### <answer>' line in a model's
    generated text.

    Rule: there must be exactly one line starting with '####', and it
    must be the last line; otherwise this counts as an extraction failure.

    Returns: (raw_answer_string, normalized_Fraction) on success,
    or (None, None) on failure.
    """
    if not isinstance(generation, str):
        return None, None

    lines = generation.rstrip().splitlines()
    marker_lines = [line for line in lines if MARKER_RE.match(line)]

    if not lines or len(marker_lines) != 1:
        return None, None
    match = FINAL_ANSWER_RE.fullmatch(lines[-1])
    if match is None:
        return None, None
    raw_answer = match.group(1)
    normalized = normalize_number(raw_answer)
    return (raw_answer, normalized) if normalized is not None else (None, None)


def fraction_to_string(value):
    """Convert a Fraction back into a readable string; integers are shown without a denominator."""
    if value.denominator == 1:
        return str(value.numerator)
    return f"{value.numerator}/{value.denominator}"


def get_eos_ids(model, tokenizer):
    eos_ids = model.generation_config.eos_token_id
    if eos_ids is None:
        eos_ids = tokenizer.eos_token_id
    if isinstance(eos_ids, int):
        eos_ids = [eos_ids]
    else:
        eos_ids = list(eos_ids)

    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    if im_end_id is not None and im_end_id != tokenizer.unk_token_id and im_end_id not in eos_ids:
        eos_ids.append(im_end_id)
    return eos_ids


In [ ]:
# Quick checks: run this cell before any GPU training.
# Passing means the deterministic parts of the assignment are ready.
assert normalize_number("1,200") == Fraction(1200)
assert normalize_number("-3.5") == Fraction(-7, 2)
assert normalize_number("3/4") == Fraction(3, 4)
assert normalize_number("1/0") is None
assert extract_final_answer("Work\n#### 42") == ("42", Fraction(42))
assert extract_final_answer("#### 42\nextra") == (None, None)
assert extract_final_answer("#### 42\n#### 43") == (None, None)
print("Parser checks passed. Run the metrics check after the evaluation helper cell.")


## Evaluation

The shared evaluation helper stores generated text and numerical outcomes for each example. Greedy decoding uses at most 256 new tokens and batches of eight.


In [ ]:
def compute_metrics(predictions):
    n = len(predictions)
    n_extracted = sum(bool(p["extraction_success"]) for p in predictions)
    n_correct = sum(bool(p["correct"]) and bool(p["extraction_success"]) for p in predictions)
    return {
        "n_examples": n,
        "accuracy": n_correct / n if n else 0.0,
        "extraction_success_rate": n_extracted / n if n else 0.0,
        "conditional_accuracy": n_correct / n_extracted if n_extracted else 0.0,
        "n_extracted": n_extracted,
        "n_correct": n_correct,
    }

def evaluate(model, tokenizer, dataset, condition, benchmark, max_new_tokens,
             batch_size=1, device="cuda", progress_path=None, run_hash=None):
    model.eval()
    tokenizer.padding_side = "left"
    eos_ids = get_eos_ids(model, tokenizer)
    predictions, previous_seconds, previous_peak = [], 0.0, 0.0
    if progress_path and os.path.exists(progress_path):
        with open(progress_path, encoding="utf-8") as handle:
            progress = json.load(handle)
        assert progress["run_hash"] == run_hash, "Evaluation checkpoint belongs to a different protocol."
        predictions = progress["predictions"]
        previous_seconds = progress["wall_clock_seconds"]
        previous_peak = progress["peak_gpu_memory_gb"]
        assert len(predictions) <= len(dataset)
        for index, record in enumerate(predictions):
            item = dataset.data[index]
            expected_id = item.get("example_id", item.get("ID"))
            assert record["example_id"] == expected_id
            assert record["source"] == item["source"] and record["gold_answer"] == item["target"]
            assert record["condition"] == condition and record["benchmark"] == benchmark
        print(f"Resume {condition}/{benchmark}: {len(predictions)}/{len(dataset)}")

    from torch.utils.data import Subset
    loader = DataLoader(Subset(dataset, range(len(predictions), len(dataset))),
                        batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    use_cuda = str(device).startswith("cuda")
    if use_cuda:
        torch.cuda.synchronize(device)
        torch.cuda.reset_peak_memory_stats(device)
    started = time.perf_counter()
    for batch in tqdm(loader, desc=f"{condition}/{benchmark}"):
        encoded = tokenizer(batch["source"], return_tensors="pt",
                            padding=True, add_special_tokens=False).to(device)
        with torch.inference_mode():
            output_ids = model.generate(
                **encoded, max_new_tokens=max_new_tokens, do_sample=False,
                eos_token_id=eos_ids, pad_token_id=tokenizer.pad_token_id,
            )
        raw_outputs = tokenizer.batch_decode(
            output_ids[:, encoded["input_ids"].shape[1]:], skip_special_tokens=True
        )
        for example_id, source, target, raw in zip(
            batch["example_id"], batch["source"], batch["target"], raw_outputs
        ):
            answer, value = extract_final_answer(raw)
            _, gold = extract_final_answer(target)
            if gold is None:
                raise ValueError(f"Unparseable gold answer: {benchmark}/{example_id}")
            predictions.append({
                "example_id": example_id, "source": source, "raw_output": raw,
                "extracted_answer": answer, "extraction_success": value is not None,
                "gold_answer": target, "correct": value is not None and value == gold,
                "condition": condition, "benchmark": benchmark,
            })
        if use_cuda:
            torch.cuda.synchronize(device)
        elapsed = previous_seconds + time.perf_counter() - started
        peak = max(previous_peak, torch.cuda.max_memory_allocated(device) / 1024**3 if use_cuda else 0.0)
        if progress_path:
            atomic_json(progress_path, {
                "run_hash": run_hash, "predictions": predictions,
                "wall_clock_seconds": elapsed, "peak_gpu_memory_gb": peak,
            })
    if use_cuda:
        torch.cuda.synchronize(device)
    metrics = compute_metrics(predictions)
    metrics.update({
        "condition": condition, "benchmark": benchmark, "run_hash": run_hash,
        "wall_clock_seconds": previous_seconds + time.perf_counter() - started,
        "peak_gpu_memory_gb": max(previous_peak, torch.cuda.max_memory_allocated(device) / 1024**3 if use_cuda else 0.0),
    })
    assert metrics["n_examples"] == len(dataset)
    print(metrics)
    return predictions, metrics

assert compute_metrics([])["accuracy"] == 0.0
_check = compute_metrics([
    {"extraction_success": True, "correct": True},
    {"extraction_success": True, "correct": False},
    {"extraction_success": False, "correct": False},
])
assert _check["accuracy"] == 1/3 and _check["conditional_accuracy"] == 1/2
print("Metrics checks passed.")


In [ ]:
import pandas as pd

all_predictions, all_metrics = {}, []
def save_predictions_jsonl(predictions, condition, benchmark, out_dir):
    path = os.path.join(out_dir, f"{condition}_{benchmark}_predictions.jsonl")
    temporary = path + ".tmp"
    with open(temporary, "w", encoding="utf-8") as handle:
        for record in predictions:
            handle.write(json.dumps(record, ensure_ascii=False, allow_nan=False) + "\n")
    os.replace(temporary, path)
    return path

def load_predictions_jsonl(path):
    with open(path, encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

def run_and_save(model, tokenizer, ds, condition, benchmark, **kwargs):
    run = {
        "condition": condition, "benchmark": benchmark,
        "model": MODEL_ID, "model_revision": MODEL_REVISION,
        "code_hash": CODE_HASH, "packages": PACKAGE_VERSIONS, "split_hash": SPLIT_HASH,
        "prompt": INSTRUCTION, "decoding": {"do_sample": False, **kwargs},
        "data_hash": digest(ds.data),
        "adapter_hash": None if condition == "original" else file_sha256(
            os.path.join(CKPT_DIR, f"{condition}_lora.pt")
        ),
    }
    run_hash = digest(run)
    pred_path = os.path.join(SUBMIT_DIR, f"{condition}_{benchmark}_predictions.jsonl")
    metrics_path = os.path.join(SUBMIT_DIR, f"{condition}_{benchmark}_metrics.json")
    progress_path = os.path.join(LOG_DIR, f"{condition}_{benchmark}_progress.json")
    if os.path.exists(metrics_path) and os.path.exists(pred_path):
        with open(metrics_path, encoding="utf-8") as handle:
            metrics = json.load(handle)
        assert metrics["run_hash"] == run_hash, "Saved results use a different configuration."
        preds = load_predictions_jsonl(pred_path)
        assert len(preds) == len(ds)
        assert all(metrics[k] == v for k, v in compute_metrics(preds).items())
        print("Reusing completed evaluation:", condition, benchmark)
    else:
        atomic_json(os.path.join(LOG_DIR, f"{condition}_{benchmark}_config.json"), run)
        preds, metrics = evaluate(model, tokenizer, ds, condition, benchmark,
                                  progress_path=progress_path, run_hash=run_hash, **kwargs)
        save_predictions_jsonl(preds, condition, benchmark, SUBMIT_DIR)
        atomic_json(metrics_path, metrics)
    all_predictions[(condition, benchmark)] = preds
    all_metrics.append(metrics)
    return preds, metrics

benchmarks = [("gsm8k", gsm8k_test_dataset_default), ("svamp", svamp_dataset)]
for benchmark_name, ds in benchmarks:
    run_and_save(lm, tokenizer, ds, condition="original", benchmark=benchmark_name,
                 max_new_tokens=MAX_NEW_TOKENS, batch_size=EVAL_BATCH_SIZE, device="cuda")


## Hook-based LoRA

Trainable low-rank branches are attached to frozen linear layers with PyTorch forward hooks. The implementation includes the attention projections, MLP projections, and language-model head.


In [ ]:
class LoRAHook:
    """
    Implements LoRA (Low-Rank Adaptation) as a forward hook.

    Instead of modifying the original nn.Linear layer's structure, this
    registers a forward hook that adds a low-rank branch (B @ A) on top
    of the layer's output after it has been computed:

        output = base_layer(x) + scaling * (x @ A) @ B

    A and B are two trainable matrices, much smaller than the original
    weight matrix (hence "low-rank"). The base_layer itself stays frozen
    and is never updated during training.
    """
    def __init__(self, module, rank=8, alpha=16, dropout=0.1):
        """
        Initialize a LoRA hook to be attached to a given nn.Linear layer.

        Args:
            module: The original layer to inject LoRA into (typically
                     nn.Linear or a quantized Linear4bit layer). This
                     function does not freeze its parameters; freezing is
                     handled by the external training code.
            rank: Rank r of the low-rank matrices. Smaller rank means
                  fewer trainable parameters.
            alpha: Numerator of the scaling factor; the final scaling
                   applied is alpha / rank.
            dropout: Dropout probability applied to the input of the
                     LoRA branch.
        """
        super().__init__()
        self.base_layer = module
        self.rank = rank
        self.alpha = alpha
        self.in_features = self.base_layer.in_features
        self.out_features = self.base_layer.out_features
        self.dropout = nn.Dropout(dropout)
        device = next(module.parameters()).device
        self.handle = None
        self.lora_A = nn.Parameter(torch.empty(self.in_features, rank, device=device, dtype=torch.float32))
        self.lora_B = nn.Parameter(torch.zeros(rank, self.out_features, device=device, dtype=torch.float32))
        nn.init.kaiming_uniform_(self.lora_A, a=5**0.5)
        self.scaling = alpha / rank
        # Initialize A randomly and B at zero: unchanged base output, nonzero B gradients.
        # Requirements:
        #   1. lora_A has shape (in_features, rank), lora_B has shape (rank, out_features)
        #   2. Initialize them so that at the start of training, the LoRA
        #      branch's output is exactly 0 (i.e. it doesn't change the
        #      original model's behavior yet), while still being able to
        #      receive gradients and be updated during training.
        #   3. Think about it: why can't both matrices be initialized to
        #      all zeros? Why can't both be initialized randomly?
        # self.lora_A = ...
        # self.lora_B = ...
        # Initialization
        # Solution:
        # self.lora_A = nn.Parameter(torch.zeros(self.in_features, rank, device=device))
        # self.lora_B = nn.Parameter(torch.zeros(rank, self.out_features, device=device))
        # nn.init.kaiming_uniform_(self.lora_A, a=5**0.5)
        # nn.init.zeros_(self.lora_B)
        # self.scaling = alpha / rank


    def __call__(self, module, input, output):
        """
        The forward hook callback, automatically invoked right after
        base_layer.forward() has been executed.

        Args:
            module: The layer that triggered this hook (i.e. base_layer itself).
            input: The original input passed to base_layer.forward, as a
                   tuple;
            output: The raw output of base_layer.forward(x), before any
                    LoRA contribution is added.

        Returns:
            The new output with the LoRA low-rank branch added on top;
            this automatically replaces the original output.
        """
        x = input[0]
        self.dropout.train(module.training)
        lora_x = self.dropout(x).to(self.lora_A.dtype)
        lora_out = (lora_x @ self.lora_A) @ self.lora_B * self.scaling
        return output + lora_out.to(output.dtype)



    def register(self):
        self.handle = self.base_layer.register_forward_hook(self)
        return self

    def remove(self):
        if self.handle is not None:
            self.handle.remove()
            self.handle = None


def inject_lora(model, target_modules=None, rank=8, alpha=16, dropout=0.1):
    """
    Walk through all submodules of the model and attach a LoRAHook to
    every nn.Linear layer whose name matches target_modules.

    Args:
        model: The model to inject LoRA into.
        target_modules: A list of layer names to target (e.g.
                          ["q_proj", "v_proj"]). Pass None to attach LoRA
                          to every nn.Linear layer in the model.
        rank: Rank of the low-rank matrices, passed to each LoRAHook.
        alpha: Scaling numerator, passed to each LoRAHook.
        dropout: Dropout probability for the LoRA branch, passed to each LoRAHook.

    Returns:
        A dict mapping each layer's full path name to its LoRAHook instance.
    """
    lora_hooks = {}
    for name, module in model.named_modules():
        if not isinstance(module, nn.Linear):
            continue
        leaf_name = name.split(".")[-1]
        if target_modules is not None and leaf_name not in target_modules:
            continue
        hook = LoRAHook(module, rank=rank, alpha=alpha, dropout=dropout).register()
        lora_hooks[name] = hook
    return lora_hooks

def detach_lora(lora_hooks):
    """detach all LoRA hooks, reset model's behavior"""
    for hook in lora_hooks.values():
        hook.remove()

def save_lora_adapter(lora_hooks, path, rank, alpha, dropout, target_modules):
    """
    Save only lora_A/lora_B and the corresponding hyperparameter config,
    without touching the base model's weights.
    lora_hooks: the dict {layer_name: LoRAHook instance} returned by inject_lora
    """
    state = {
        "config": {
            "rank": rank,
            "alpha": alpha,
            "dropout": dropout,
            "target_modules": target_modules,
        },
        "weights": {
            name: {
                "lora_A": hook.lora_A.detach().cpu(),
                "lora_B": hook.lora_B.detach().cpu(),
            }
            for name, hook in lora_hooks.items()
        },
    }
    torch.save(state, path)

def freeze_and_collect_trainable_params(model, lora_hooks):
    for p in model.parameters():
        p.requires_grad = False

    trainable_params = []
    for hook in lora_hooks.values():
        hook.lora_A.requires_grad = True
        hook.lora_B.requires_grad = True
        trainable_params.extend([hook.lora_A, hook.lora_B])
    return trainable_params

def load_lora_weights(lora_hooks, state, device="cuda"):
    """
    state: the full dict returned by torch.load(path) (contains "config" and "weights")
    lora_hooks: hooks that have already been injected, with a structure matching `state`
    """
    for name, hook in lora_hooks.items():
        assert name in state["weights"], f"权重文件里缺少这一层: {name}"
        hook.lora_A.data.copy_(state["weights"][name]["lora_A"].to(device))
        hook.lora_B.data.copy_(state["weights"][name]["lora_B"].to(device))


def load_lora_adapter(model, path, device="cuda"):
    """
    Starting from scratch: inject the LoRA structure into a clean model,
    then load the trained weights into it.
    """
    state = torch.load(path, map_location="cpu", weights_only=True)
    config = state["config"]

    lora_hooks = inject_lora(
        model,
        target_modules=config["target_modules"],
        rank=config["rank"],
        alpha=config["alpha"],
        dropout=config["dropout"],
    )
    load_lora_weights(lora_hooks, state, device=device)
    return lora_hooks

def infer_lora_config(lora_hooks):
    """
    Infer the LoRA hyperparameters (rank, alpha, dropout, target module
    names) directly from an already-injected set of LoRAHooks, instead
    of requiring the caller to pass them in separately (and risk them
    getting out of sync with the actual hooks).
    """
    if not lora_hooks:
        raise ValueError("lora_hooks is empty, cannot infer config")

    sample_hook = next(iter(lora_hooks.values()))
    return {
        "rank": sample_hook.rank,
        "alpha": sample_hook.alpha,
        "dropout": sample_hook.dropout.p,
        "target_modules": sorted({name.split(".")[-1] for name in lora_hooks.keys()}),
    }


In [ ]:
def check_lora_hook_init(rank=4, alpha=8, in_features=16, out_features=16):
    """
    Sanity check for LoRAHook.__init__: verifies shapes, that lora_A is
    not all-zero, that the LoRA branch's contribution is exactly zero
    right after initialization, and that the hooked layer's actual
    output matches the original (un-hooked) layer's output.
    """
    layer = nn.Linear(in_features, out_features)
    x = torch.randn(2, in_features)
    original_output = layer(x)   # capture the output BEFORE the hook is registered

    hook = LoRAHook(layer, rank=rank, alpha=alpha)

    assert hook.lora_A.shape == (in_features, rank), \
        f"lora_A should have shape {(in_features, rank)}, got {tuple(hook.lora_A.shape)}"
    assert hook.lora_B.shape == (rank, out_features), \
        f"lora_B should have shape {(rank, out_features)}, got {tuple(hook.lora_B.shape)}"
    assert not torch.allclose(hook.lora_A, torch.zeros_like(hook.lora_A)), \
        "lora_A should NOT be all-zero (otherwise gradients can never reach lora_B)"
    assert torch.allclose(hook.lora_A @ hook.lora_B, torch.zeros_like(hook.lora_A @ hook.lora_B)), \
        "lora_A @ lora_B should be exactly zero right after initialization"

    hook.register()
    hooked_output = layer(x)   # now the hook is active; this should still match original_output

    assert torch.allclose(hooked_output, original_output, atol=1e-5), \
        "Right after initialization, the hooked layer's output should be identical to the original layer's output"

    hook.remove()
    print("LoRAHook sanity check passed")


check_lora_hook_init()


## Training conditions

Both adapters use rank 8, alpha 16, dropout 0.1, two epochs, a 512-token sequence cap, and AdamW at 1e-5. Each condition starts from the same frozen base and uses the same training and validation questions.


In [ ]:
def atomic_torch_save(path, state):
    temporary = str(path) + ".tmp"
    torch.save(state, temporary)
    os.replace(temporary, path)

def train_lora(model, tokenizer, lora_hooks, train_dataset, val_dataset,
               num_epochs=2, batch_size=2, learning_rate=1e-5, device="cuda",
               max_len=512, condition=None):
    if condition is None:
        raise ValueError("Provide a training condition for the checkpoint names.")
    trainable_params = freeze_and_collect_trainable_params(model, lora_hooks)
    n_base = model.num_parameters() if hasattr(model, "num_parameters") else sum(p.numel() for p in model.parameters())
    n_trainable = sum(p.numel() for p in trainable_params)
    n_total = n_base + n_trainable
    config = {
        "condition": condition, "seed": SEED, "num_epochs": num_epochs,
        "max_seq_length": max_len, "learning_rate": learning_rate,
        "effective_batch_size": batch_size, "optimizer": "AdamW",
        "optimizer_weight_decay": 0.01, "scheduler": "none",
        "gradient_accumulation_steps": 1, "loss_aggregation": "mean_of_batch_losses",
        "lora": infer_lora_config(lora_hooks), "n_train_examples": len(train_dataset),
        "n_val_examples": len(val_dataset), "model_revision": MODEL_REVISION,
        "split_hash": SPLIT_HASH, "code_hash": CODE_HASH, "packages": PACKAGE_VERSIONS,
    }
    fingerprint = digest(config)
    resume_path = os.path.join(CKPT_DIR, f"{condition}_resume.pt")
    history_path = os.path.join(LOG_DIR, f"{condition}_train_history.json")
    optimizer = torch.optim.AdamW(trainable_params, lr=learning_rate)
    history = {"train_loss": [], "val_loss": []}
    first_epoch, first_batch, accumulated_loss = 0, 0, 0.0
    previous_seconds, previous_peak = 0.0, 0.0
    if os.path.exists(resume_path):
        state = torch.load(resume_path, map_location="cpu", weights_only=True)
        assert state["fingerprint"] == fingerprint, "Training checkpoint configuration differs."
        load_lora_weights(lora_hooks, state, device=device)
        optimizer.load_state_dict(state["optimizer"])
        history = state["history"]
        first_epoch, first_batch = state["next_epoch"], state["next_batch"]
        accumulated_loss = state["accumulated_loss"]
        previous_seconds, previous_peak = state["wall_clock_seconds"], state["peak_gpu_memory_gb"]
        torch.set_rng_state(state["torch_rng"])
        random.setstate(state["python_rng"])
        if str(device).startswith("cuda"):
            torch.cuda.set_rng_state_all(state["cuda_rng"])
        print(f"Resume {condition}: epoch={first_epoch + 1}, batch={first_batch}")
    else:
        seed_all()

    use_cuda = str(device).startswith("cuda")
    if use_cuda:
        torch.cuda.synchronize(device)
        torch.cuda.reset_peak_memory_stats(device)
    started = time.perf_counter()
    old_use_cache = model.config.use_cache
    model.config.use_cache = False

    def checkpoint(next_epoch, next_batch, accumulated):
        if use_cuda:
            torch.cuda.synchronize(device)
        atomic_torch_save(resume_path, {
            "fingerprint": fingerprint, "next_epoch": next_epoch, "next_batch": next_batch,
            "accumulated_loss": accumulated, "history": history,
            "optimizer": optimizer.state_dict(),
            "weights": {name: {"lora_A": h.lora_A.detach().cpu(),
                              "lora_B": h.lora_B.detach().cpu()} for name, h in lora_hooks.items()},
            "torch_rng": torch.get_rng_state(), "python_rng": random.getstate(),
            "cuda_rng": torch.cuda.get_rng_state_all() if use_cuda else [],
            "wall_clock_seconds": previous_seconds + time.perf_counter() - started,
            "peak_gpu_memory_gb": max(previous_peak, torch.cuda.max_memory_allocated(device) / 1024**3 if use_cuda else 0.0),
        })

    try:
        for epoch in range(first_epoch, num_epochs):
            generator = torch.Generator().manual_seed(SEED + epoch)
            train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                                      collate_fn=collate_fn, generator=generator)
            val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                                    collate_fn=collate_fn,
                                    generator=torch.Generator().manual_seed(SEED))
            model.train()
            optimizer.zero_grad(set_to_none=True)
            total_train_loss = accumulated_loss if epoch == first_epoch else 0.0
            skip = first_batch if epoch == first_epoch else 0
            for step, batch in enumerate(tqdm(train_loader, desc=f"{condition} epoch {epoch+1}/{num_epochs}")):
                if step < skip:
                    continue
                outputs = model(input_ids=batch["input_ids"].to(device),
                                attention_mask=batch["attention_mask"].to(device),
                                labels=batch["labels"].to(device))
                loss = outputs.loss
                if not torch.isfinite(loss):
                    raise FloatingPointError(f"Non-finite loss: {condition}, epoch={epoch}, batch={step}")
                loss.backward()
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                total_train_loss += loss.item()
                if (step + 1) % 100 == 0:
                    checkpoint(epoch, step + 1, total_train_loss)
            history["train_loss"].append(total_train_loss / len(train_loader))
            model.eval()
            total_val_loss = 0.0
            with torch.no_grad():
                for batch in tqdm(val_loader, desc=f"{condition} validation"):
                    loss = model(input_ids=batch["input_ids"].to(device),
                                 attention_mask=batch["attention_mask"].to(device),
                                 labels=batch["labels"].to(device)).loss
                    if not torch.isfinite(loss):
                        raise FloatingPointError("Non-finite validation loss.")
                    total_val_loss += loss.item()
            history["val_loss"].append(total_val_loss / len(val_loader))
            print(f"Epoch {epoch+1}: train={history['train_loss'][-1]:.5f}, val={history['val_loss'][-1]:.5f}")
            checkpoint(epoch + 1, 0, 0.0)
            atomic_json(history_path, {**config, **history, "status": "in_progress"})
    finally:
        model.config.use_cache = old_use_cache

    if use_cuda:
        torch.cuda.synchronize(device)
    history.update(config)
    history.update({
        "wall_clock_seconds": previous_seconds + time.perf_counter() - started,
        "peak_gpu_memory_gb": max(previous_peak, torch.cuda.max_memory_allocated(device) / 1024**3 if use_cuda else 0.0),
        "n_base_params": n_base, "n_total_params": n_total, "n_trainable_params": n_trainable,
        "trainable_param_pct": 100 * n_trainable / n_total,
        "trainable_param_fraction": n_trainable / n_total,
        "lora_rank": config["lora"]["rank"], "lora_alpha": config["lora"]["alpha"],
        "lora_dropout": config["lora"]["dropout"], "lora_target_modules": config["lora"]["target_modules"],
        "status": "complete", "fingerprint": fingerprint,
    })
    return history

def train_condition(condition, train_dataset, val_dataset):
    seed_all()
    adapter_path = os.path.join(CKPT_DIR, f"{condition}_lora.pt")
    history_path = os.path.join(LOG_DIR, f"{condition}_train_history.json")
    hooks = inject_lora(lm, target_modules=LORA_TARGET_MODULES, rank=LORA_RANK,
                        alpha=LORA_ALPHA, dropout=LORA_DROPOUT)
    try:
        history = None
        if os.path.exists(history_path):
            with open(history_path, encoding="utf-8") as handle:
                history = json.load(handle)
        if os.path.exists(adapter_path) and history and history.get("status") == "complete":
            assert history["num_epochs"] == NUM_EPOCHS
            assert history["code_hash"] == CODE_HASH and history["model_revision"] == MODEL_REVISION
            assert history["adapter_sha256"] == file_sha256(adapter_path)
            print("Reusing completed adapter:", condition)
        else:
            if os.path.exists(adapter_path):
                assert os.path.exists(os.path.join(CKPT_DIR, f"{condition}_resume.pt")), "Cannot recover an incomplete adapter without a training checkpoint."
            history = train_lora(
                lm, tokenizer, hooks, train_dataset, val_dataset, num_epochs=NUM_EPOCHS,
                batch_size=TRAIN_BATCH_SIZE, learning_rate=LEARNING_RATE, max_len=MAX_SEQ_LENGTH,
                device="cuda", condition=condition,
            )
            save_lora_adapter(hooks, adapter_path, rank=LORA_RANK, alpha=LORA_ALPHA,
                              dropout=LORA_DROPOUT, target_modules=LORA_TARGET_MODULES)
            history["adapter_sha256"] = file_sha256(adapter_path)
            atomic_json(history_path, history)
        print(history)
        return history
    finally:
        detach_lora(hooks)
        gc.collect()
        torch.cuda.empty_cache()


In [ ]:
assert (NUM_EPOCHS, MAX_SEQ_LENGTH, LEARNING_RATE) == (2, 512, 1e-5)
assert (LORA_RANK, LORA_ALPHA, LORA_DROPOUT) == (8, 16, 0.1)
assert (TRAIN_BATCH_SIZE, EVAL_BATCH_SIZE, MAX_NEW_TOKENS) == (2, 8, 256)
print("Required configuration confirmed.")


In [ ]:
history_answer = train_condition("sft_answer", gsm8k_train_dataset_answer_only, gsm8k_val_dataset_answer_only)


In [ ]:
history_reasoning = train_condition("sft_reasoning", gsm8k_train_dataset_reasoning, gsm8k_val_dataset_reasoning)


## Adapter evaluations

Both trained adapters are evaluated on GSM8K test and complete SVAMP. Every generation batch is checkpointed for resumability.


In [ ]:
# =========================================================
# Run 4 LoRA evaluations
# =========================================================

# Just in case you restarted the kernal:
if "all_predictions" not in dir():
    all_predictions = {}
if "all_metrics" not in dir():
    all_metrics = []
if "benchmarks" not in dir():
    benchmarks = [("gsm8k", gsm8k_test_dataset_default), ("svamp", svamp_dataset)]

# ---- SFT-Answer ----
lora_hooks = load_lora_adapter(lm, CKPT_DIR + "/sft_answer_lora.pt", device="cuda")
for benchmark_name, ds in benchmarks:
    run_and_save(lm, tokenizer, ds, condition="sft_answer", benchmark=benchmark_name,
                 max_new_tokens=MAX_NEW_TOKENS, batch_size=EVAL_BATCH_SIZE, device="cuda")
detach_lora(lora_hooks)
del lora_hooks
gc.collect()
torch.cuda.empty_cache()

# ---- SFT-Reasoning ----
lora_hooks = load_lora_adapter(lm, CKPT_DIR + "/sft_reasoning_lora.pt", device="cuda")
for benchmark_name, ds in benchmarks:
    run_and_save(lm, tokenizer, ds, condition="sft_reasoning", benchmark=benchmark_name,
                 max_new_tokens=MAX_NEW_TOKENS, batch_size=EVAL_BATCH_SIZE, device="cuda")
detach_lora(lora_hooks)
del lora_hooks
gc.collect()
torch.cuda.empty_cache()


## Main results

Six saved metric files populate the results table. The generalization gap is GSM8K accuracy minus SVAMP accuracy.


In [ ]:
# =========================================================
# Prepare score.csv
# =========================================================
import os
import json
import pandas as pd

# WORK_DIR was set once in the setup cell.
SUBMIT_DIR = os.path.join(WORK_DIR, "submission")

conditions = ["original", "sft_answer", "sft_reasoning"]
benchmark_names = ["gsm8k", "svamp"]

all_metrics = []
for condition in conditions:
    for benchmark_name in benchmark_names:
        metrics_path = os.path.join(SUBMIT_DIR, f"{condition}_{benchmark_name}_metrics.json")
        pred_path = os.path.join(SUBMIT_DIR, f"{condition}_{benchmark_name}_predictions.jsonl")

        assert os.path.exists(metrics_path), f"metrics missing: {metrics_path}"
        assert os.path.exists(pred_path), f"predictions missing: {pred_path}"

        with open(metrics_path, encoding="utf-8") as f:
            metrics = json.load(f)
        all_metrics.append(metrics)

assert len(all_metrics) == 6, f"6 records required，but have {len(all_metrics)}"

scores = pd.DataFrame(all_metrics)

accuracy_lookup = scores.set_index(["condition", "benchmark"])["accuracy"].to_dict()
gaps = {c: accuracy_lookup[(c, "gsm8k")] - accuracy_lookup[(c, "svamp")] for c in conditions}
baseline_accuracy = {b: accuracy_lookup[("original", b)] for b in benchmark_names}

scores["generalization_gap_svamp"] = scores["condition"].map(gaps)
scores["accuracy_change_vs_original"] = scores.apply(
    lambda row: row["accuracy"] - baseline_accuracy[row["benchmark"]], axis=1
)

scores = scores[[
    "condition", "benchmark", "n_examples", "accuracy", "extraction_success_rate",
    "conditional_accuracy", "generalization_gap_svamp", "accuracy_change_vs_original",
    "n_extracted", "n_correct", "wall_clock_seconds", "peak_gpu_memory_gb",
]].sort_values(["condition", "benchmark"])

scores_path = os.path.join(SUBMIT_DIR, "scores.csv")
scores.to_csv(scores_path, index=False)
display(scores)
print(f"saved: {scores_path}")


In [ ]:
required_fields = {"example_id", "raw_output", "extracted_answer", "extraction_success",
                   "gold_answer", "correct", "condition", "benchmark"}
for condition in conditions:
    for benchmark, dataset in benchmarks:
        path = os.path.join(SUBMIT_DIR, f"{condition}_{benchmark}_predictions.jsonl")
        rows = load_predictions_jsonl(path)
        assert len(rows) == len(dataset)
        assert len({r["example_id"] for r in rows}) == len(dataset)
        for index, row in enumerate(rows):
            assert required_fields.issubset(row)
            item = dataset.data[index]
            assert row["example_id"] == item.get("example_id", item.get("ID"))
            assert row["source"] == item["source"] and row["gold_answer"] == item["target"]
            answer, value = extract_final_answer(row["raw_output"])
            _, gold = extract_final_answer(row["gold_answer"])
            assert row["extracted_answer"] == answer
            assert row["extraction_success"] == (value is not None)
            assert row["correct"] == (value is not None and value == gold)
            assert row["condition"] == condition and row["benchmark"] == benchmark
        with open(os.path.join(SUBMIT_DIR, f"{condition}_{benchmark}_metrics.json"), encoding="utf-8") as handle:
            metrics = json.load(handle)
        assert all(metrics[k] == v for k, v in compute_metrics(rows).items())
for condition in ("sft_answer", "sft_reasoning"):
    with open(os.path.join(LOG_DIR, f"{condition}_train_history.json"), encoding="utf-8") as handle:
        history = json.load(handle)
    assert history["status"] == "complete" and len(history["train_loss"]) == NUM_EPOCHS
assert len(scores) == 6
print("All six result files and training histories verified.")


## Paired error analysis

Predictions are compared by example ID to count improvements, regressions, and format changes. Optional comparisons are run with the scripts in `optional/`.


In [ ]:
# Paired comparisons use only the saved, automatically scored predictions.
paired_summary = []
for benchmark, dataset in benchmarks:
    by_condition = {
        condition: load_predictions_jsonl(os.path.join(SUBMIT_DIR, f"{condition}_{benchmark}_predictions.jsonl"))
        for condition in conditions
    }
    for condition in ("sft_answer", "sft_reasoning"):
        baseline, adapted = by_condition["original"], by_condition[condition]
        paired_summary.append({
            "benchmark": benchmark, "condition": condition,
            "wrong_to_correct": sum(not a["correct"] and b["correct"] for a, b in zip(baseline, adapted)),
            "correct_to_wrong": sum(a["correct"] and not b["correct"] for a, b in zip(baseline, adapted)),
            "extraction_failure_to_success": sum(not a["extraction_success"] and b["extraction_success"] for a, b in zip(baseline, adapted)),
        })
    # Save the first example in each observed outcome category; do not edit outputs.
    cases = {}
    for records in zip(*(by_condition[c] for c in conditions)):
        outcome = tuple((r["extraction_success"], r["correct"]) for r in records)
        cases.setdefault(str(outcome), {c: r for c, r in zip(conditions, records)})
    atomic_json(os.path.join(ANALYSIS_DIR, f"{benchmark}_paired_cases.json"), cases)
paired_table = pd.DataFrame(paired_summary)
paired_table.to_csv(os.path.join(ANALYSIS_DIR, "paired_changes.csv"), index=False)
display(paired_table)
print("Error-analysis candidates saved:", ANALYSIS_DIR)


## Run artifacts

New runs save scores and per-example predictions under `submission/`, training histories and provenance under `logs/`, and adapter checkpoints under `ckpt/`. Historical measurements are available in the repository's `results/recorded/` directory.


## Recorded results

`notebooks/results_review.ipynb` reads the exported tables without loading a model. The training code in this notebook has portable paths and fixed upstream revisions; its outputs are generated when the pipeline is run.


## Reproduction scope

The recorded run used one RTX 5090 and training seed 42. Repeated seeds can assess the stability of the observed differences. Package versions, input splits, and upstream revisions are recorded with the results.
